In [1]:
# 変化量の上位95%だけを抽出
import pandas as pd
import os

# 入力ファイル
input_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"

# 出力ファイル
output_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δint-95.csv"

# 出力先が無ければ新規作成
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# CSV読み込み
df = pd.read_csv(input_csv)

filter_95 = df[df["delta_int_kushinada"] > 2.94]

result = filter_95[
    [
        "filename",
        "text",
        "valence",
        "arousal",
        "dominance",
        "intensity_pred_kushinada",
        "delta_valence",
        "delta_arousal",
        "delta_dominance",
        "delta_int_kushinada",
    ]
]

# 保存
result.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_csv}")
print(f"発話数: {len(result)}")

保存完了: /home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δint-95.csv
発話数: 13


In [2]:
df[df["delta_int_kushinada"] > 2.94]

,filename,text,valence,arousal,dominance,intensity_pred_kushinada,delta_valence,delta_arousal,delta_dominance,delta_int_kushinada
18,0044_R.wav,多分(F あのー)多分ポイントポイントを話した方がいいんだろうなとは思ったんですけど,0.49,0.52,0.49,0.09,0.04,0.00,0.02,4.10
26,0065_R.wav,(F はい),0.42,0.33,0.34,0.63,0.19,0.03,0.07,3.12
39,0086_R.wav,(F うー)聞いていたので知識はあったから,0.40,0.42,0.44,-1.08,0.00,0.06,0.07,2.97
42,0096_R.wav,<笑>,0.74,0.63,0.58,4.71,0.19,0.03,0.00,4.40
65,0140_R.wav,(F いや)そうでもないです全く,0.64,0.60,0.55,1.20,0.28,0.49,0.36,3.27
80,0163_R.wav,先行ってた,0.35,0.47,0.46,-1.52,0.51,0.26,0.24,3.78
126,0261_R.wav,(F ん)だから今日も送っていった後に来てそれで京浜東北線で(D ぬ)ずっと来たからあんな時...,0.68,0.56,0.56,0.43,0.30,0.14,0.19,3.09
160,0335_R.wav,<笑>,0.95,0.80,0.77,5.82,0.32,0.17,0.15,3.98
161,0336_R.wav,<笑>,0.88,0.77,0.73,2.85,0.07,0.03,0.04,2.97
167,0347_R.wav,その辺がやっぱり大変かなと,0.35,0.29,0.31,0.03,0.00,0.17,0.11,2.99


In [4]:
# パーセンタイル95%のRの発話
import pandas as pd
import os
import shutil

# ==========================
# パス
# ==========================
l_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/eval-int"

os.makedirs(output_root, exist_ok=True)

# ==========================
# CSV読込（Lのみ）
# ==========================
df_l = pd.read_csv(l_csv)

# 発話番号（CSVのインデックスではなく、0015_L.wav の「15」を指定）
change_idx = [
    44, 65, 86, 96, 140, 163, 261, 335, 336, 347, 353, 428, 458
]

context = 3

# ==========================
# 抽出
# ==========================
for utt_no in change_idx:

    # 発話番号からファイル名を作成
    target_filename = f"{utt_no:04d}_R.wav"

    # filenameから該当行を検索
    matched = df_l[df_l["filename"] == target_filename]

    if matched.empty:
        print(f"{target_filename} がCSV内に見つかりません")
        continue

    # DataFrame内でのインデックス取得
    idx = matched.index[0]

    print(f"発話番号 {utt_no} -> DataFrame index {idx}")

    # 前後3発話（Lのみ）
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Utterance number : {utt_no}\n")
        f.write(f"DataFrame index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 音声コピー
    for _, row in context_df.iterrows():

        src = os.path.join(wav_dir, row["filename"])

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # ターゲット音声にはTARGET_を付ける
        if row["filename"] == target_filename:
            dst_name = "TARGET_" + row["filename"]
        else:
            dst_name = row["filename"]

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

発話番号 44 -> DataFrame index 18
発話番号 65 -> DataFrame index 26
発話番号 86 -> DataFrame index 39
発話番号 96 -> DataFrame index 42
発話番号 140 -> DataFrame index 65
発話番号 163 -> DataFrame index 80
発話番号 261 -> DataFrame index 126
発話番号 335 -> DataFrame index 160
発話番号 336 -> DataFrame index 161
発話番号 347 -> DataFrame index 167
発話番号 353 -> DataFrame index 170
発話番号 428 -> DataFrame index 211
発話番号 458 -> DataFrame index 224
完了
